# **Import des modules nécessaires**

In [14]:
import pandas as pd #pour la manipulation de données
import numpy as np
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from spacy.lang.fr.stop_words import STOP_WORDS as spacy_stopwords

from bertopic import BERTopic #pour la modélisation de sujets
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic
from sentence_transformers import SentenceTransformer #pour les embeddings de phrase

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from sklearn.metrics import silhouette_score

from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel


# **Chargement du corpus de Zola et de spacy**


In [15]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",737
1,1865 La confession de Claude.,1865,1,2,"Pars cependant, puisque tu as soif de la vie. ...",584
2,1865 La confession de Claude.,1865,1,3,Le premier rayon a chassé le cauchemar de ma v...,731
3,1865 La confession de Claude.,1865,1,4,Hélas! il me faut cependant une ombre de réali...,574
4,1865 La confession de Claude.,1865,1,5,Ce corps frissonnant veillé par cette horrible...,744


In [16]:
df.shape

(5327, 6)

# **Traitement du Corpus de Zola**

In [17]:
stop_perso = {
    "grand", "petit", "homme", "femme", "jour", "heure", "coup", "œil", "oeil", 
    "main", "bras", "tête", "voix", "milieu", "eau", "terre", "air", "monde", 
    "chose", "nuit", "vie", "enfant", "père", "mère", "fille", "garçon", 
    "monsieur", "madame", "falloir", "aller", "voir", "dire", "faire", 
    "pouvoir", "vouloir", "savoir", "venir", "devoir", "prendre", "donner",
    "oui", "non", "où", "quand", "comment", "bon", "jeune", "vieux", "suite"
}

# Chargement du modèle avec désactivation du 'parser' syntaxique pour gagner en vitesse
# On garde impérativement 'ner' pour repérer les personnages/lieux et 'lemmatizer'
nlp = spacy.load("fr_core_news_lg", disable=["parser"])
nlp.max_length = 2_000_000  

#on convertit en liste
textes_bruts = df["texte"].astype(str).tolist()

textes_nettoyes = []

# Utilisation de nlp.pipe pour traiter les textes par blocs (très rapide)
for doc in nlp.pipe(textes_bruts, batch_size=256, n_process=2): 
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules
        
        if (
            not token.is_stop # Ignorer les stop words spaCy par défaut
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces vides
            and token.ent_type_ not in ['PER', 'LOC', 'ORG'] # Ignorer les Personnages, Lieux et Organisations
            and token.pos_ in {"NOUN", "ADJ"}  # Garder Noms, Adjectifs ET Verbes
            and len(lemme) > 2 # Ignorer les mots de 1 ou 2 lettres
            and lemme not in stop_perso
        ):
            tokens.append(lemme)
            
    # Rejoindre les tokens validés et les ajouter à la liste finale
    textes_nettoyes.append(" ".join(tokens))

# Application de la liste nettoyée à la nouvelle colonne du DataFrame
df["phrases_lemm"] = textes_nettoyes

chemin_sortie = Path("..") /"data" /"2_processed" /"03_corpus_lematise.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(chemin_sortie, index=False, encoding="utf-8")

# Affichage du résultat
df[["phrases_lemm"]].head()

,phrases_lemm
0,hiver matin frais manteau brouillard saison so...
1,soif projet soi ferme loyal action rêve vis gr...
2,rayon cauchemar veille obstacle lutte désert c...
3,ombre réalité inquiet joie conscience vague pl...
4,corps horrible rapide impression dégoût effroi...


Si on a deja lemmatisé le jeu de données on peut directement charger le ficher csv 

In [18]:
df=pd.read_csv(Path("..")/"data" /"2_processed"/"03_corpus_lematise.csv", encoding="utf-8")

## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [19]:
embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(),batch_size=64, show_progress_bar=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Génération des embeddings sémantiques...


Batches:   0%|          | 0/84 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

### 1) HDBSCAN et UMAP

### 3) CountVectorizer et ClassTfidfTransformer avec des stop words personnalisés 

In [20]:
hdbscan_model = HDBSCAN( min_cluster_size=27, 
                        min_samples=4, 
                        metric='euclidean', 
                        cluster_selection_method='eom',
                        prediction_data=True)

umap_model = UMAP( n_neighbors=20,
                  n_components=3, 
                  min_dist=0.0, 
                  metric="cosine",
                  random_state=42)


vectorizer_model = CountVectorizer(
    min_df=2,    # Le mot doit apparaître dans au moins 2 segments pour être pris en compte (élimine les fautes ou mots uniques)
    max_df=0.6)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)


topic_model = BERTopic(
    language="french",
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True,
    #nr_topics="auto"
)
topics, probs = topic_model.fit_transform(df["phrases_lemm"].tolist(), embeddings= embeddings)

new_topics = topic_model.reduce_outliers(
    df["phrases_lemm"].tolist(), 
    topics, 
    strategy="embeddings",
    embeddings=embeddings
)

# Met à jour le modèle avec ces nouveaux thèmes plus propres
topic_model.update_topics(df["phrases_lemm"].tolist(), topics=new_topics)

2026-07-26 16:19:53,734 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-26 16:19:58,679 - BERTopic - Dimensionality - Completed ✓
2026-07-26 16:19:58,679 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-26 16:19:58,749 - BERTopic - Cluster - Completed ✓
2026-07-26 16:19:58,750 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-26 16:19:58,962 - BERTopic - Representation - Completed ✓
2026-07-26 16:19:59,127 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


## **3) Topics Présent**

In [21]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,483,0_cœur_amour_mort_chambre,"[cœur, amour, mort, chambre, soir, fond, fois,...",[instant examen cri supplication seigneur mala...
1,1,414,1_haut_soleil_noir_blanc,"[haut, soleil, noir, blanc, ciel, long, fond, ...",[génération ruine droit trouvaille conquête ca...
2,2,211,2_peuple_travail_justice_science,"[peuple, travail, justice, science, bonheur, œ...",[sort société solution paisible question socia...
3,3,278,3_abbé_affaire_prêtre_porte,"[abbé, affaire, prêtre, porte, ami, maison, ch...",[mesure long attente grâce prêtre faute -vou é...
4,4,308,4_franc_maison_soir_fils,"[franc, maison, soir, fils, frère, dernier, af...",[maître puddleur four halle laminoir rail pudd...
5,5,243,5_face_pied_sang_lit,"[face, pied, sang, lit, porte, gros, chambre, ...",[place froid côté dormant lourd bourdonnant po...
6,6,268,6_dame_maison_mari_porte,"[dame, maison, mari, porte, cher, salon, abbé,...",[vision modèle bénitier tante brancardier proc...
7,7,251,7_gros_porte_table_bout,"[gros, porte, table, bout, rire, fois, fond, p...",[rencontre train bouteille genièvre débit hasa...
8,8,253,8_maison_porte_chambre_long,"[maison, porte, chambre, long, face, fond, foi...",[glorieux existence cou fortune désastre villa...
9,9,231,9_porte_dame_maison_chambre,"[porte, dame, maison, chambre, affaire, moment...",[mariage démarche semaine question banquette t...


In [22]:
# 1. Préparer les données pour Gensim (une liste de listes de mots)
# On suppose que df["phrases_lemm"] contient tes textes nettoyés
textes_tokenises = [texte.split() for texte in df["phrases_lemm"].tolist()]
dictionnaire = Dictionary(textes_tokenises)
corpus = [dictionnaire.doc2bow(texte) for texte in textes_tokenises]

# 2. Extraire les mots-clés des topics trouvés par BERTopic
# On ignore le topic -1 s'il existe
topics_mots = []
for topic_id in set(topics):
    if topic_id != -1:
        # Récupère juste les mots, pas les scores
        mots = [mot[0] for mot in topic_model.get_topic(topic_id)]
        topics_mots.append(mots)

# 3. Calculer le score de cohérence (C_v)
cm = CoherenceModel(
    topics=topics_mots, 
    texts=textes_tokenises, 
    corpus=corpus, 
    dictionary=dictionnaire, 
    coherence='c_v'
)
coherence_score = cm.get_coherence()
print(f"Score de Cohérence (C_v) : {coherence_score:.4f}")

Score de Cohérence (C_v) : 0.4548


In [23]:
# On récupère les documents qui NE SONT PAS dans le topic -1 (outliers)
# (Si tu as utilisé reduce_outliers, tous les documents auront un topic valid)
indices_valides = [i for i, topic in enumerate(topics) if topic != -1]

# On filtre les embeddings et les topics
embeddings_valides = np.array([embeddings[i] for i in indices_valides])
topics_valides = [topics[i] for i in indices_valides]

# Calcul du score
score = silhouette_score(embeddings_valides, topics_valides)
print(f"Score de Silhouette : {score:.3f}")

Score de Silhouette : 0.007


In [24]:
fig = topic_model.visualize_hierarchy()
fig.show()

In [25]:
# Récupération de la dimension temporelle
timestamps = df['annee'].tolist()

# Génération des topics dans le temps
topics_over_time = topic_model.topics_over_time(
    df['phrases_lemm'].tolist(), 
    timestamps, 
    nr_bins=15
)

topic_model.visualize_topics_over_time(topics_over_time) #topics=themes_interet)

15it [00:01,  8.83it/s]


In [26]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('cœur', np.float64(0.014550020348965666)), ('amour', np.float64(0.013313393159504293)), ('mort', np.float64(0.01237906794309681)), ('chambre', np.float64(0.01081198044002358)), ('soir', np.float64(0.009901197668827005)), ('fond', np.float64(0.009877617446102539)), ('fois', np.float64(0.009154472993961072)), ('joie', np.float64(0.009111568271428155)), ('dernier', np.float64(0.008888545098600566)), ('heureux', np.float64(0.008827943174322566))]

TOPIC 1
[('haut', np.float64(0.011997376084195753)), ('soleil', np.float64(0.010697617800287704)), ('noir', np.float64(0.010129805979969606)), ('blanc', np.float64(0.010054534867716645)), ('ciel', np.float64(0.009824504415450375)), ('long', np.float64(0.009648271985775326)), ('fond', np.float64(0.009549752970797222)), ('ombre', np.float64(0.008546117024338891)), ('mort', np.float64(0.00853724210317148)), ('plein', np.float64(0.008340107690337193))]

TOPIC 2
[('peuple', np.float64(0.01859268650526085)), ('travail', np.float64(0.01369553